## 0. Imports and data loading

In this homework we will use the classic **MNIST** dataset of handwritten digits (28×28, grayscale).


In [4]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

print("TensorFlow version:", tf.__version__)

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Load MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Add channel dim: (N, 28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

num_classes = 10
input_shape = x_train.shape[1:]

print("Train shape:", x_train.shape, "Test shape:", x_test.shape)


TensorFlow version: 2.19.0
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train shape: (60000, 28, 28, 1) Test shape: (10000, 28, 28, 1)


## Task 1: Basic TensorFlow tensors and operations

Implement the function `tensor_stats(x)`.

**Requirements:**

- Input: 1D `tf.Tensor` of shape `(n,)` with `dtype=tf.float32`.
- Return: a Python `dict` with keys:
  - `"mean"` — mean value
  - `"std"` — standard deviation
  - `"min"` — minimum
  - `"max"` — maximum
- All values in the dict must be **Python floats** (not tensors).
- Use **TensorFlow ops only**:
  - `tf.reduce_mean`, `tf.math.reduce_std`, `tf.reduce_min`, `tf.reduce_max`
- Do **not** convert `x` to NumPy inside the function.


In [5]:
import tensorflow as tf

def tensor_stats(x: tf.Tensor) -> dict:
    """Return basic statistics for a 1D float32 tensor."""
    mean = tf.reduce_mean(x)
    std = tf.math.reduce_std(x)
    min_v = tf.reduce_min(x)
    max_v = tf.reduce_max(x)

    return {
        "mean": float(mean.numpy()),
        "std": float(std.numpy()),
        "min": float(min_v.numpy()),
        "max": float(max_v.numpy())
    }

## Task 2: Dense neural network for MNIST with `tf.keras`

Implement the function `build_dense_mnist_model(input_shape, num_classes)`.

**Requirements:**

- Use **`tf.keras` / `keras` only**.
- Input: images of shape `input_shape` (e.g. `(28, 28, 1)`).
- Output: probabilities over `num_classes` classes (use `softmax`).
- Architecture:
  - Flatten layer
  - At least **one Dense hidden layer** with ≥ 128 units and **ReLU** activation
  - Final Dense layer with `num_classes` units and **softmax** activation
- The model must be **compiled** with:
  - Optimizer: `Adam` with learning rate `1e-3`
  - Loss: `sparse_categorical_crossentropy`
  - Metric: `accuracy`


In [7]:
from tensorflow import keras
from tensorflow.keras import layers

def build_dense_mnist_model(input_shape, num_classes):
    """Build and compile a dense neural network for MNIST."""
    inputs = keras.Input(shape=input_shape)
    x = keras.layers.Flatten()(inputs)
    x = keras.layers.Dense(128, activation="relu")(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

## Task 3: `tf.data` pipeline for MNIST

Implement the function `make_mnist_dataset(x, y, batch_size)`.

**Requirements:**

- Create a `tf.data.Dataset` from NumPy arrays `x` and `y`.
- Shuffle the dataset with a buffer size ≥ `len(x) // 4`.
- Batch with the given `batch_size`.
- Prefetch with `tf.data.AUTOTUNE`.
- Return the prepared dataset.


In [9]:
SEED = 42

def make_mnist_dataset(x, y, batch_size: int) -> tf.data.Dataset:
    """Create a shuffled, batched, prefetched tf.data.Dataset."""
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    buffer_size = max(len(x) // 4, 1)
    ds = ds.shuffle(buffer_size=buffer_size, seed=SEED)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

## Task 4: Custom training step with `tf.GradientTape`

Implement the function `train_one_epoch(model, dataset, optimizer, loss_fn)` that performs **one epoch** of training
with a custom loop.

**Requirements:**

- Iterate over `(x_batch, y_batch)` from `dataset`.
- Use `tf.GradientTape()` to compute gradients of the loss w.r.t. **trainable variables**.
- Apply gradients via `optimizer.apply_gradients(...)`.
- Accumulate the mean loss for the epoch and return it as a Python float.


In [11]:
def train_one_epoch(model, dataset, optimizer, loss_fn) -> float:
    """Run one epoch of custom training loop."""
    total_loss = 0.0
    num_batches = 0

    for x_batch, y_batch in dataset:
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)
            loss_value = loss_fn(y_batch, logits)

        grads = tape.gradient(loss_value, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        total_loss += float(loss_value.numpy())
        num_batches += 1

    if num_batches == 0:
        return 0.0
    return total_loss / num_batches